# Football Action Coach — Google Colab quickstart

## Before you start

1. **Runtime → Change runtime type → GPU** (T4 or better). CPU will be very slow for YOLO + ViTPose.
2. **OpenCV player selection does not work in Colab.** `preprocess` / `infer` open a desktop window and wait for a mouse click. In Colab you should either:
   - Run **preprocess on your own machine**, then zip `data/<video_id>/` and upload to Google Drive; mount Drive below and copy into `data/`, **or**
   - Run **only `train`** here on data you already prepared.
3. **Checkpoints** (`checkpoints/vitpose_huge.pth`, SAM2 if used) are large — download with `wget` or upload from Drive (not stored on GitHub).
4. **mmpose configs**: `PoseEstimator` uses relative config paths from the **mmpose** package layout. If `init_model` fails, set `vitpose_config` in `config.py` to the absolute path of `ViTPose_huge_coco_256x192.py` inside your `mmpose` install (search site-packages), or run from a cloned [mmpose](https://github.com/open-mmlab/mmpose) repo per their docs.

Repo: [github.com/tomerzi/sam-anlys](https://github.com/tomerzi/sam-anlys)

In [ ]:
# Clone from GitHub (HTTPS)
%cd /content
!rm -rf sam-anlys 2>/dev/null
!git clone https://github.com/tomerzi/sam-anlys.git
%cd sam-anlys/football_action_coach
!pwd && ls -la

In [ ]:
# Core Python deps (matches requirements.txt pip-installable lines)
!pip install -q torch torchvision ultralytics supervision xgboost scikit-learn joblib scipy opencv-python

In [ ]:
# mmpose stack for ViTPose (can take several minutes)
!pip install -qU openmim
!mim install -q mmengine "mmcv>=2.0.0" mmdet mmpose

### Optional: SAM 2

SAM2 is optional (`config.use_sam2`). To skip it on Colab, in `config.py` set `use_sam2: bool = False` (faster setup). To enable, install from the [SAM2 repo](https://github.com/facebookresearch/sam2) and download `sam2_hiera_large.pt` into `checkpoints/` per `requirements.txt` comments.

In [ ]:
# Example: download ViTPose-H checkpoint into checkpoints/
import os
os.makedirs("checkpoints", exist_ok=True)
VIT_URL = (
    "https://download.openmmlab.com/mmpose/v1/body_2d_keypoint/topdown_heatmap/coco/"
    "td-hm_ViTPose-huge_8xb64-210e_coco-256x192-e32adcd4_20230314.pth"
)
!wget -nc -O checkpoints/vitpose_huge.pth "$VIT_URL"

In [ ]:
# Optional Colab tweaks: skip SAM2 (faster), or force CPU if CUDA errors
from pathlib import Path

cfg = Path("config.py")
text = cfg.read_text()
text = text.replace("use_sam2: bool = True", "use_sam2: bool = False")
# text = text.replace('device: str = "cuda"', 'device: str = "cpu"')
cfg.write_text(text)
print("Patched config: use_sam2=False. Uncomment CPU line in this cell if needed.")

### Mount Google Drive (optional)

If you preprocessed on your PC, zip `data/` (and optionally `models/`) and upload to Drive. Then:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Example: copy prepared data into the project
# !cp -r "/content/drive/MyDrive/sam_anlys_data/data" .
# !cp -r "/content/drive/MyDrive/sam_anlys_data/models" . 2>/dev/null || true

### Run commands

All commands assume current directory is `.../football_action_coach` (set by the first clone cell).

```bash
# Train (needs ≥2 videos under data/<id>/)
python main.py train

# Infer (needs trained models/ — interactive click will NOT work in Colab)
python main.py infer --video /path/to/video.mp4
```

Preprocess (usually **local PC only**):

```bash
python main.py preprocess --video clip.mp4 --label 2 --id my_run_good
```

Labels: `0` = bad, `1` = ok, `2` = good.

In [ ]:
%cd /content/sam-anlys/football_action_coach
# After data/ and models/ are in place:
# !python main.py train
# !python main.py infer --video /content/your_clip.mp4